In [ ]:
!pip install -q onnxruntime

In [ ]:
import json
import sys

import torch
from huggingface_hub import snapshot_download
from tokenizers import Tokenizer

model_dir = snapshot_download('quark28/mini-llm-classic-sft')
sys.path.insert(0, model_dir)

from sampler import generate, OrtBackend, PAD_ID, USER_ID, BOT_ID, EOS_ID

config = json.load(open(f'{model_dir}/config.json'))
settings = config['SAMPLER_SETTINGS']
tok = Tokenizer.from_file(f'{model_dir}/tokenizer.json')
model = OrtBackend(f'{model_dir}/onnx/encoder.onnx', f'{model_dir}/onnx/decoder.onnx')

In [ ]:
CONTEXT_LEN = 256


def trim(ids):
    if len(ids) <= CONTEXT_LEN:
        return ids
    for i, t in enumerate(ids):
        if t == USER_ID and len(ids) - i <= CONTEXT_LEN:
            return ids[i:]
    return ids[-CONTEXT_LEN:]


history = []
print("mini-llm classic SFT. Type 'exit' to stop.")
while (message := input('USER: ')) != 'exit':
    history += [USER_ID] + tok.encode(message).ids + [BOT_ID]
    ids = trim(history)
    ids = [PAD_ID] * (CONTEXT_LEN - len(ids)) + ids
    out = generate(model, torch.tensor(ids, dtype=torch.long), device='cpu', **settings).tolist()
    reply = out[:out.index(EOS_ID)]
    history += reply + [EOS_ID]
    print(f' BOT: {tok.decode(reply)}')